In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"D:\code\data\green\ppg_final_features.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\code\\data\\green\\ppg_final_features.csv'

In [ ]:
df["hr"] = (6e4 / df.rri).astype(int)

In [ ]:
df.columns = ["timestamp", "rri", "areaUp", "areaDown", "motion", "hr"]

In [ ]:
df

In [ ]:
dff = df[df.hr > 40]

In [ ]:
dff.hr.hist(bins=100)

In [ ]:
m = dff.motion.to_numpy()
plt.figure(figsize=(18, 5))
plt.scatter(np.arange(len(m)), m)
plt.show()

In [ ]:
# 将 motion 列全体减 300，并将所有小于 0 的值限制为 0
dff['motion'] = (dff['motion'] - 300).clip(lower=0)

In [ ]:
dff = dff[(dff.areaUp <= 1e5) & (dff.areaDown <= 3e5) & (dff.motion < 1500)]
dff 

In [ ]:
dff.areaUp.hist(bins=100)

In [ ]:
# 1. 获取当前 areaUp 列的最小值和最大值
min_val = dff['areaUp'].min()
max_val = dff['areaUp'].max()

# 2. 执行 Min-Max 缩放 (映射到 0 - 5000 范围)
if max_val > min_val:  # 防止整列数字完全一样导致除以 0 的报错
    dff['areaUp'] = (dff['areaUp'] - min_val) / (max_val - min_val) * 5000
else:
    dff['areaUp'] = 0  # 如果全列数据一样，直接重置为 0（或者填 5000 也可以）

# 如果你需要确保最后的结果是整数 (比如为了后续的 Hex 编码)
# df['areaUp'] = df['areaUp'].round().astype(int)

In [ ]:
dff[dff.areaUp == 0]

In [ ]:
# 1. 获取当前 areaUp 列的最小值和最大值
min_val = dff['areaDown'].min()
max_val = dff['areaDown'].max()

# 2. 执行 Min-Max 缩放 (映射到 0 - 5000 范围)
if max_val > min_val:  # 防止整列数字完全一样导致除以 0 的报错
    dff['areaDown'] = (dff['areaDown'] - min_val) / (max_val - min_val) * 1e4
else:
    dff['areaDown'] = 0  # 如果全列数据一样，直接重置为 0（或者填 5000 也可以）

# 如果你需要确保最后的结果是整数 (比如为了后续的 Hex 编码)
# df['areaUp'] = df['areaUp'].round().astype(int)

In [ ]:
dff.areaDown.hist(bins=100)

In [ ]:
dff.to_csv("fea_manual.csv", index=False)

In [ ]:
dff.dtypes

In [ ]:
dff.motion.hist(bins=100)

In [ ]:
dff.to_csv(r"D:\code\data\green\filtered_ppg_final_features.csv", index=False)

In [ ]:
import pickle


In [ ]:
fp = r"D:\code\feature_extract_pipeline\target\_target_dist_cache.pkl"
with open(fp, "rb") as f:
    target_dist_cache = pickle.load(f)

In [ ]:
tar = target_dist_cache

In [ ]:
tar.keys()

In [ ]:
plt.hist(tar["area_up"], bins=200)
plt.show()

In [ ]:
plt.hist(tar["area_down"], bins=200)
plt.show()

In [ ]:
plt.hist(tar["motion"], bins=200)
plt.show()

In [ ]:
dff.head()

In [ ]:
rep = r"D:\code\data\green\filtered_ppg_final_features_adapted.csv"
rf = pd.read_csv(rep)

In [ ]:
rf

In [ ]:
rf.areaDown_DA.hist(bins=200)

In [ ]:
rf.motion_DA.hist(bins=200)

In [ ]:
df.dtypes

In [ ]:

# 假设你的 DataFrame 变量名为 df

# 1. 将字符串格式的时间戳转换为 Datetime 格式
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. 按分钟 ('1min') 对数据进行重采样并求均值
# 做法是将 timestamp 设为索引，重采样后再把索引还原成列
df_minute_avg = df.set_index('timestamp').resample('1min').mean().reset_index()

# 如果你更喜欢在一行内完成，也可以使用 pd.Grouper：
# df_minute_avg = df.groupby(pd.Grouper(key='timestamp', freq='1min')).mean().reset_index()



In [ ]:
df_minute_avg

In [ ]:
df.motion.hist(bins=100)

In [ ]:
m = df_minute_avg.motion.to_numpy()
plt.plot(m)
plt.show()

In [ ]:
fi = rf[["timestamp", "rri", "areaUp_DA", "areaDown_DA", "motion_DA"]]
fi.columns = ["timestamp", 'rri', 'areaUp', 'areaDown', 'motion']

In [ ]:
fi.to_csv("onepass_feature.csv", index=False)

In [ ]:
fi.dtypes